<a href="https://colab.research.google.com/github/GODZBREAKER/Machine-Translation-Systems-Architectures-and-LLM-Adaptation/blob/main/FGAI_Assignment3_Code_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ICT4403 - Foundations of Generative AI
## Assignment 3: Machine Translation (Google Colab Version)
**Dataset:** Multi30k (English to German)
**Models:** Basic Transformer, mT5, LLaMA, Gemma

In [ ]:
!pip install -q transformers datasets evaluate sacrebleu accelerate peft bitsandbytes
!pip install -q torch torchvision torchaudio huggingface_hub

## 0. Hugging Face Authentication (Required for Gemma)
Run this cell and paste your Hugging Face Access Token.
**Note:** You must first visit [google/gemma-2b](https://huggingface.co/google/gemma-2b) on Hugging Face and accept the license agreement.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 1. Memory Management Utility
Colab's free T4 GPU has 16GB of VRAM. This utility helps clear memory between model evaluations.

In [ ]:
import gc
import torch

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print("CUDA VRAM cleared.")
    else:
        print("RAM cleared.")

free_memory()

RAM cleared.


## 2. Dataset Preparation

In [ ]:
from datasets import load_dataset

dataset = load_dataset("bentrevett/multi30k")
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']
print(f"Train size: {len(train_data)}")

README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/29000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1014 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train size: 29000


## 3. Basic Transformer

In [ ]:
import torch.nn as nn

class BasicTransformer(nn.Module):
    def __init__(self, d_model=256, nhead=4, num_encoder_layers=2, num_decoder_layers=2):
        super().__init__()
        self.transformer = nn.Transformer(d_model, nhead, num_encoder_layers, num_decoder_layers, batch_first=True)

    def forward(self, src, tgt):
        return self.transformer(src, tgt)

print("Basic Transformer initialized.")
free_memory()

Basic Transformer initialized.
RAM cleared.


## 4. mT5 (Fine-Tuning)

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, TrainingArguments

mt5_name = "google/mt5-small"
mt5_tokenizer = AutoTokenizer.from_pretrained(mt5_name, legacy=False)
mt5_model = AutoModelForSeq2SeqLM.from_pretrained(mt5_name, device_map="auto")

training_args = TrainingArguments(
    output_dir="./mt5_results",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=4,
    fp16=True,
    save_steps=500,
    max_steps=1000,
)
print("mT5 loaded.")
# del mt5_model
# free_memory()

config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.20GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

... (2 outputs left out)

model.safetensors: downloading bytes:           |  0.00B            

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

mT5 loaded.


## 5. LLaMA (4-Bit Quantization)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

llama_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
llama_tokenizer = AutoTokenizer.from_pretrained(llama_name)

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

llama_model = AutoModelForCausalLM.from_pretrained(
    llama_name,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
print(f"{llama_name} loaded in 4-bit.")
# del llama_model
# free_memory()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

TinyLlama/TinyLlama-1.1B-Chat-v1.0 loaded in 4-bit.


## 6. Gemma (4-Bit Quantization)
This cell will now work correctly as long as you logged in using `notebook_login()` in Cell 0.

In [ ]:
gemma_name = "google/gemma-2b"

gemma_tokenizer = AutoTokenizer.from_pretrained(gemma_name)
gemma_model = AutoModelForCausalLM.from_pretrained(
    gemma_name,
    quantization_config=quant_config,
    device_map="auto",
    low_cpu_mem_usage=True
)
print(f"{gemma_name} loaded successfully in 4-bit.")

config.json:   0%|          | 0.00/627 [00:00<?, ?B/s]

[transformers] We found `hidden_act="gelu"` in this Gemma config. This is a legacy value of the official releases but it is meant to target the tanh approximation. Setting `hidden_act="gelu_pytorch_tanh"` instead.


tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.5MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/164 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

google/gemma-2b loaded successfully in 4-bit.


## 7. Evaluation

In [ ]:
import evaluate

bleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

preds = ["Guten Morgen"]
refs = [["Guten Morgen"]]

bleu_score = bleu.compute(predictions=preds, references=refs)
chrf_score = chrf.compute(predictions=preds, references=refs)

print(f"BLEU: {bleu_score['score']:.2f}")
print(f"chrF: {chrf_score['score']:.2f}")

BLEU: 0.00
chrF: 100.00
